# The Deduction Solver

This notebook documents the `Solver` class, in which we apply deduction to get the board to a state where no more moves can be made with certainty.

## Outline

1. [How solve() works](#1-how-solve()-works)
2. [The Frontier](#2-the-frontier)
3. [Pattern 1: Single-Point Rule](#3-pattern-1--single-point-rule)
4. [Pattern 2: Subset Rule](#4-pattern-2--subset-rule)
5. [Running examples on real boards](#5-running-examples-on-real-boards)
6. [Limitations](#6-limitations)
7. [Summary & Next Phase](#7-summary-&-next-phase)

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import random
from src.game.board import Board, HIDDEN, KNOWN_MINE, print_board
from src.solver import Solver

## 1. How solve() works

`solve()` simply runs on a loop, which applies our pattern methods until nothing changes, meaning that we have found all the patterns we can.

```
while changed and not board.game_over:
    changed = False
    frontier = board.frontier()
    if _single_point(board, frontier):  
        changed = True
    if _subset_rule(board, frontier):   
        changed = True
```

After the loop exits, the solver inspects the board state and returns:

| Return value | Meaning |
|---|---|
| `"won"` | Every safe cell was revealed by forced moves (unlikely to happen at our target board config)|
| `"stuck"` | Deduction is exhausted, guesses are needed |
| `"lost"` | A forced reveal hit a mine (should never happen) |

As described in the previous notebook, we make sure that we never use the `is_mine()` function, to make sure that this uses purely deduction and doesn't cheat.

Surprisingly, we only need two patterns to cover most (not exactly all) situations which simply need deductions. For example, the common 1-2-1 pattern is encapsulated in multiple of the two patterns we have implemented. This makes the implementation much simpler than expected.

## 2. The Frontier

Both patterns iterate through the **frontier**. We define it as the set of revealed cells that still have a hidden neighbour, meaning that they still have to be resovled. 

This is computed by `board.frontier()`, so can be easily passed into our pattern methods.

## 3. Pattern 1: Single-Point Rule

This the simplest pattern. For a revealed cell with value `k`, let:
- `remaining` = `k` minus the number of adjacent cells already marked as known mines
- `hidden` = the list of adjacent cells that are still hidden

Two cases produce forced moves:

| Condition | Meaning | Action |
|---|---|---|
| `remaining == 0` | All adjacent mines are already marked | All `hidden` neighbors are **safe** so they can be revealed |
| `remaining == len(hidden)` | Every hidden neighbor must be a mine | Mark all `hidden` neighbors as **known mines** |

This is fast (O(n), where n is the size of the frontier) and handles the vast majority of
moves on any board.

### Example: all neighbors safe

A cell shows `1`, and its single mine is already marked.
`remaining = 1 - 1 = 0` so we reveal every hidden neighbor.

In [9]:
s = Solver()

# 4x4 board, mine at (0,1) — only known fact for this demo
b = Board(4, 4, mine_count=1, rng=random.Random(0))

# Manually set up a mid-game state:
# (1,1) shows 1, its only mine (0,1) is already marked
# so remaining=0 → (0,2) is safe
b._mine_grid[0][1] = True
b._generated = True
b._adj_counts[0][1] = -1
for r in range(4):
    for c in range(4):
        if b._mine_grid[r][c]:
            b._adj_counts[r][c] = -1
        else:
            b._adj_counts[r][c] = sum(
                1 for nr, nc in b.neighbors(r, c) if b._mine_grid[nr][nc]
            )

# Reveal just (1,1) and (0,0) to create the scenario
b._player_grid[1][1] = b._adj_counts[1][1]  # shows 1
b._player_grid[0][0] = b._adj_counts[0][0]  # shows 1
b._player_grid[0][2] = b._adj_counts[0][2]  # shows 1
b._revealed_count = 3
b.mark_known_mine(0, 1)

print_board(b)
print(f"  remaining at (1,1) = {b.adj_remaining_mines(1,1)}")
print(f"  hidden neighbors of (1,1) = {b.unrevealed_neighbors(1,1)}")

    0  1  2  3
    ------------
 0 | 1  *  1  . 
 1 | .  1  .  . 
 2 | .  .  .  . 
 3 | .  .  .  . 
  remaining at (1,1) = 0
  hidden neighbors of (1,1) = [(1, 0), (1, 2), (2, 0), (2, 1), (2, 2)]


### Example: all neighbors are mines

A cell shows `2` with exactly 2 hidden neighbors, no known mines adjacent.
`remaining == len(hidden) == 2` → both neighbors are mines.

In [10]:
b2 = Board(4, 4, mine_count=4, rng=random.Random(0))
mines = {(1,2), (2,1), (2,3), (3,2)}
for r, c in mines:
    b2._mine_grid[r][c] = True
b2._generated = True
for r in range(4):
    for c in range(4):
        if b2._mine_grid[r][c]:
            b2._adj_counts[r][c] = -1
        else:
            b2._adj_counts[r][c] = sum(
                1 for nr, nc in b2.neighbors(r, c) if b2._mine_grid[nr][nc]
            )

# Reveal a few cells to create the scenario
for r, c, v in [(0,0,0),(0,1,0),(0,2,0),(0,3,0),
                (1,0,0),(1,1,1),(1,3,1),
                (2,0,0),(2,2,3),(2,3,1),(3,3,1)]:
    b2._player_grid[r][c] = v
    b2._revealed_count += 1

print_board(b2)
# Show the constraint
print(f"  remaining at (2,2) = {b2.adj_remaining_mines(2,2)}")
print(f"  hidden neighbors of (2,2) = {b2.unrevealed_neighbors(2,2)}")

    0  1  2  3
    ------------
 0 | 0  0  0  0 
 1 | 0  1  .  1 
 2 | 0  .  3  1 
 3 | .  .  .  1 
  remaining at (2,2) = 4
  hidden neighbors of (2,2) = [(1, 2), (2, 1), (3, 1), (3, 2)]


## 4. Pattern 2: Subset Rule

Single-point only looks at one cell at a time. The subset rule reasons
about **pairs** of revealed cells.

Suppose revealed cells A and B satisfy:
- A's hidden neighbors: $S_A = \{x_1, x_2, \dots\}$
- B's hidden neighbors: $S_B = \{x_1, x_2, \dots\}$
- $S_A \subset S_B$ (strict subset)

Then A's mines are entirely contained within B's region.
The extra cells $S_B \setminus S_A$ must account for the difference:

$$\text{mines in } S_B \setminus S_A \;=\; b - a$$

| Condition | Meaning | Action |
|---|---|---|
| $b - a = 0$ | The extra cells carry no mines | Reveal every cell in $S_B \setminus S_A$ |
| $b - a = \lvert S_B \setminus A \rvert$ | All extra cells are mines | Mark every cell in $S_B \setminus S_A$ |

As we said, many Minesweeper patterns are just the subset rule applied
repeatedly:

- **1-2 pattern**: the 1's hidden set is a subset of the 2's.
- **1-2-1**: two applications of the subset rule, one from each 1 to the 2.
- **1-2-2-1**: same idea, four pairwise comparisons.

You don't need separate methods for each of these patterns, the subset rule
handles them all.

In [11]:
# Concrete board example of subset rule in action
# 3x3 with 2 mines: (1,2) and (2,1)

b3 = Board(3, 3, mine_count=2, rng=random.Random(0))
mines3 = {(1,2), (2,1)}
for r, c in mines3:
    b3._mine_grid[r][c] = True
b3._generated = True
for r in range(3):
    for c in range(3):
        if b3._mine_grid[r][c]:
            b3._adj_counts[r][c] = -1
        else:
            b3._adj_counts[r][c] = sum(
                1 for nr, nc in b3.neighbors(r, c) if b3._mine_grid[nr][nc]
            )

# Reveal (0,0), (0,1), (0,2), (1,0), (1,1)
# (0,0)=1, (0,1)=1, (0,2)=1, (1,0)=1, (1,1)=2
for r, c, v in [(0,0,1),(0,1,1),(0,2,1),(1,0,1),(1,1,2)]:
    b3._player_grid[r][c] = v
    b3._revealed_count += 1

print_board(b3)

f = b3.frontier()
print(f"Frontier: {f}")
print()
for r, c in f:
    rem = b3.adj_remaining_mines(r, c)
    hid = b3.unrevealed_neighbors(r, c)
    print(f"  ({r},{c}): remaining={rem}, hidden={hid}")
print()

    0  1  2
    ---------
 0 | 1  1  1 
 1 | 1  2  . 
 2 | .  .  . 
Frontier: [(0, 1), (0, 2), (1, 0), (1, 1)]

  (0,1): remaining=1, hidden=[(1, 2)]
  (0,2): remaining=1, hidden=[(1, 2)]
  (1,0): remaining=1, hidden=[(2, 0), (2, 1)]
  (1,1): remaining=2, hidden=[(1, 2), (2, 0), (2, 1), (2, 2)]



## 5. Running examples on real boards

Let's create boards using the solver's natural flow — first click,
then solve.

In [12]:
# Example 1: A board where single-point + subset win completely
b_easy = Board(5, 5, mine_count=2, rng=random.Random(100))
b_easy.reveal(2, 2)

result = s.solve(b_easy)
print(f"Result: {result}")
print(f"Revealed: {b_easy.revealed_count}/{b_easy.total_cells - b_easy.mine_count} safe cells")
print(f"Known mines: {b_easy.remaining_total_mines()} remaining")
print(f"Game state: {b_easy.game_state}")
print_board(b_easy)

Result: won
Revealed: 23/23 safe cells
Known mines: 2 remaining
Game state: GameState.WON
    0  1  2  3  4
    ---------------
 0 | 0  0  0  0  0 
 1 | 0  0  0  1  1 
 2 | 0  0  0  1  . 
 3 | 1  1  0  1  1 
 4 | .  1  0  0  0 


In [13]:
# Example 2: higher mine density, more interesting deductions
b_med = Board(9, 9, mine_count=15, rng=random.Random(200))
b_med.reveal(4, 4)

result = s.solve(b_med)
print(f"Result: {result}")
print(f"Revealed: {b_med.revealed_count}/{b_med.total_cells - b_med.mine_count} safe cells")
print(f"Known mines marked: {b_med.mine_count - b_med.remaining_total_mines()}")
print_board(b_med)

Result: won
Revealed: 66/66 safe cells
Known mines marked: 15
    0  1  2  3  4  5  6  7  8
    ---------------------------
 0 | *  2  0  1  1  1  1  1  1 
 1 | *  2  0  1  *  2  2  *  1 
 2 | 1  1  0  1  2  *  2  1  1 
 3 | 1  1  0  0  1  1  2  2  2 
 4 | *  1  0  0  0  0  1  *  * 
 5 | 1  2  1  2  1  2  3  5  4 
 6 | 0  1  *  2  *  2  *  *  * 
 7 | 1  2  1  3  2  3  2  3  2 
 8 | *  1  0  1  *  1  0  0  0 


In [14]:
# Target board size: 30x30 with 270 mines
b_target = Board(30, 30, mine_count=270, rng=random.Random(3000))
b_target.reveal(15, 15)

result = s.solve(b_target)
print(f"Result: {result}")
print(f"Revealed: {b_target.revealed_count}/{b_target.total_cells - b_target.mine_count} safe cells")
print(f"Known mines marked: {b_target.mine_count - b_target.remaining_total_mines()}")
print(f"Frontier size: {len(b_target.frontier())}")
print_board(b_target)

Result: stuck
Revealed: 94/630 safe cells
Known mines marked: 34
Frontier size: 26
    0  1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29
    ------------------------------------------------------------------------------------------
 0 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 1 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 2 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 3 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 4 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 5 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 6 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 7 | .  .  .  .  .  .  .  .  .  .  .  .  .  

## 6. Limitations

Single-point and the subset rule cover the vast majority of forced moves.
But there are situations where neither rule fires, even though the board
is *not* stuck. This means we have to take a different approach.

### The limitation

Both rules are **local**:
- Single-point: looks at one revealed cell at a time.
- Subset rule: looks at **pairs** of revealed cells, but only when one's
  hidden set is a strict subset of the other's.

Neither considers three or more constraints **jointly**. When constraints
are overlapping but not nested, the forced cells can only be found by
enumerating all valid configurations and checking which cells are
determined in every one of them.

### Concrete example: a case subset misses

Consider three revealed cells with these constraints:

```
A: remaining=1, hidden neighbors = {a, b, c}
B: remaining=1, hidden neighbors = {a, b, d}
C: remaining=1, hidden neighbors = {c, d, e}
```

- `{a,b,c}` and `{a,b,d}` are **not** subsets of each other, so subset rule skips.
- `{c,d,e}` is not a subset of either, so no pair fires.
- Single-point finds no cell with `remaining==0` or `remaining==len(hidden)`.

Every valid configuration has `e=1`, meaning `e` is a **forced mine**.
The subset rule alone cannot see this because no pair of hidden sets
is nested.

### How we handle this

The way forward isn't to add more patterns which aim to handle these sort of cases. The rate of appearance of such cases isn't high enough to validate an extra pattern check here, it's unnecesary. The best way to handle these is by enumeration since we will be doing this anyway. This means that we should leave this to the probability engine. The way we will pass this on is by simply making the engine first check if any mines are guaranteed, i.e. probability 1, and if so then they can be safely revealed and essentially will be ignored in terms of taking data.

If at the simulation stage we find that our solver is too slow, and enumeration is taking too long, we can take inspiration from other known algorithms for this case and implement them to reduce the amount of enumeration we have to do.

## 7. Summary & Next Phase

Now that we have a solver which is (almost) fully able to play all forced moves, we have to be able to handle the guesses.
Let's clearly outline the divide between 

| | Solver | Probability engine |
|---|---|---|
| **Job** | Find forced cells (mines and safe cells) | Compute mine probabilities for ambiguous cells |
| **Method** | Fast pattern matching (single-point + subset) | Full enumeration of valid configurations per component |
| **Cells resolved** | Cells forced by any local rule | Cells whose probability is 0 (safe) or 1 (mine) and ambiguous cells |
| **When it runs** | When there are forced moves to be made | When the solver returns "stuck" |

The key insight: the probability engine's enumeration **is** the complete
deduction closure. When it finds a cell with probability 1.0, that cell is
a forced mine; when it finds probability 0.0, that cell is safe. So the
probability engine does everything the solver does, plus the hard cases
the solver can't reach.

This means the solver acts as a **fast path** for the common cases. Most of
the board is resolved cheaply by single-point and subset patterns. The expensive
enumeration only runs on the frontier that remains after those
fast rules are exhausted.

### What the simulation runner will do

1. First Click
2. Run solver
3. If stuck, go to probability engine (enumerate, compute probabilities)
4. If a cell has p=0 or p=1, mark/reveal (still a forced move, so ignore it)
5. If ambiguous, pick lowest probability cell to be a mine and reveal
6. Repeat 2-5 until win or loss